# 🛡️ CyberShield AI — Research Edition (Google Colab)
### Multilingual AI-Powered Phishing Detection System (Egyptian Arabic, MSA, English) & URL Inspection
نظام ذكي متعدد اللغات لكشف التصيد الاحتيالي وتحليل الروابط التفسيري

--- 
**طريقة التشغيل على Google Colab:**
1. قم بتشغيل الخلايا بالترتيب (اضغط على زر التشغيل ▶️ في كل خلية أو `Ctrl + F9` لتشغيل الكل).
2. الخلية الأولى تقوم بتثبيت المكتبات المطلوبة (`scikit-learn`, `fastapi`, `uvicorn`, إلخ).
3. الخلية الثانية تنشئ ملفات محرك المعالجة وتدرب النماذج وتستخرج المقاييس البحثية.
4. الخلية الثالثة تتيح لك الفحص التفاعلي المباشر داخل Google Colab!
5. الخلية الرابعة تشغل خادم الويب وتوفر لك رابطاً عاماً (Public URL) لفتح الموقع كاملاً!

## 1️⃣ تثبيت المتطلبات (Install Dependencies)

In [ ]:
#@title تثبيت المكتبات وتجهيز البيئة السحابية
!pip install -q fastapi uvicorn scikit-learn numpy scipy joblib pydantic pyngrok nest-asyncio
print('✅ تم تثبيت جميع المكتبات بنجاح في بيئة Google Colab!')

## 2️⃣ بناء وتشغيل محرك CyberShield AI وتدريب النماذج (Train & Initialize Models)

In [ ]:
#@title إنشاء الكود وتدريب النماذج البحثية (TF-IDF + Random Forest)
import os, sys, re, math, json, time, sqlite3
from urllib.parse import urlparse
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print('🛡️ جاري تجهيز المعمارية التقنية لـ CyberShield AI...')

# --- Arabic Normalizer ---
DIACRITICS_REGEX = re.compile(r'[\u064B-\u0652\u0653-\u0655\u0670]')
TATWEEL_REGEX = re.compile(r'\u0640')
EGYPTIAN_COLLOQUIAL_TERMS = {'كسبت', 'فزت', 'الحق', 'دوس', 'عشان', 'عشانك', 'هنقفل', 'هيتوقف', 'عاللينك', 'عالرابط', 'ابعث', 'ابعت', 'فودافون', 'انستاباي', 'كاش', 'الاهلي', 'بنك مصر', 'فيزا', 'كود السحب', 'جنيه', 'شحنة', 'كود التحقق', 'محدش يشاركه', 'فلوس'}

SUSPICIOUS_ARABIC_PATTERNS = [
    (r'(تم\s*(إيقاف|تجميد|حظر|تعليق|حسابك\s*معلق)\s*(حسابك|بطاقتك|محفظتك)?)', 2.5, 'تنبيه إيقاف أو تجميد حساب/محفظة'),
    (r'(تحديث\s*(البيانات|بياناتك|الهوية|الحساب|بطاقتك))', 2.2, 'طلب لتحديث بيانات بنكية أو بطاقة'),
    (r'(مبروك\s*(كسبت|فزت|ربحت))', 2.4, 'إغراء جوائز وهمية ومسابقات احتيالية'),
    (r'(فودافون\s*كاش|انستاباي|انستا\s*باي|محفظة\s*ذكية).*(كسبت|فزت|ايقاف|تحديث|حظر|رابط|لينك|كود|سحب|هدية|استلم|فلوس)', 2.2, 'استخدام محفظة إلكترونية مقترناً باستدراج احتيالي'),
    (r'(كود\s*(التحقق|التأكيد|OTP|السر|السري|السحب))', 1.8, 'إشارة لكود التحقق السري (OTP)'),
    (r'(دوس\s*(ع\s*اللينك|عاللينك|هنا)|اضغط\s*(هنا|ع\s*الرابط|عالرابط|عاللينك|ع\s*اللينك))', 2.2, 'حث مباشر ومكثف للضغط على رابط'),
    (r'(شحنتك\s*(معلقة|في الانتظار|لم تسلم)|سداد\s*(رسوم|مصاريف))', 2.0, 'انتحال شركات الشحن والطرود'),
    (r'(فرصة\s*(أخيرة|محدودة|لن تتكرر)|ساعات\s*قليلة\s*متبقية)', 1.5, 'خلق شعور وهمي بالعجلة والضغط الزمني')
]
LEGITIMATE_SAFETY_PATTERNS = [
    (r'(لا\s*(تشارك|تفصح|تعط)\s*(هذا\s*الرمز|الكود|كلمة\s*السر|الرقم\s*السري))', 2.0, 'تحذير رسمي بعدم مشاركة الرمز السري'),
    (r'(تم\s*(تحويل|سحب|إيداع)\s*مبلغ.*(رصيدك\s*الحالي|الرصيد\s*المتاح))', 2.0, 'إشعار معاملة بنكية رسمية برصيد حساب')
]
SUSPICIOUS_ENGLISH_PATTERNS = [
    (r'(account\s*(suspended|locked|disabled|blocked|compromised))', 2.5, 'Account suspension/blocking threat'),
    (r'(verify\s*your\s*(account|identity|credentials|card|wallet))', 2.2, 'Urgent account/identity verification request'),
    (r'(you\s*(won|have\s*been\s*selected|are\s*the\s*lucky\s*winner))', 2.2, 'Fake lottery/reward prize hook'),
    (r'(click\s*(here|on\s*the\s*link|below)|login\s*now|claim\s*now)', 1.7, 'Direct call to action to external link')
]

def normalize_arabic(text):
    if not text: return ''
    text = DIACRITICS_REGEX.sub('', text)
    text = TATWEEL_REGEX.sub('', text)
    text = re.sub(r'[إأآٱ]', 'ا', text)
    text = re.sub(r'ى\b', 'ي', text)
    text = re.sub(r'ة\b', 'ه', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return re.sub(r'\s+', ' ', text).strip()

def detect_language(text):
    ar_chars = len(re.findall(r'[\u0600-\u06FF]', text))
    en_chars = len(re.findall(r'[a-zA-Z]', text))
    total = ar_chars + en_chars
    if total == 0: return {'language': 'unknown', 'is_egyptian': False}
    words = set(normalize_arabic(text.lower()).split())
    egyptian = len(words.intersection(EGYPTIAN_COLLOQUIAL_TERMS)) > 0
    if ar_chars / total > 0.65:
        lang = 'arz' if egyptian else 'ar'
    elif en_chars / total > 0.65:
        lang = 'en'
    else:
        lang = 'mixed'
    return {'language': lang, 'is_egyptian': egyptian}

# --- URL Features ---
URL_PATTERN = re.compile(r'(?:(?:https?|ftp):\/\/)?(?:(?:[a-zA-Z0-9_\-\.]+\.[a-zA-Z]{2,})|(?:\d{1,3}(?:\.\d{1,3}){3}))(?::\d{1,5})?(?:\/[^\s\(\)\[\]\{\}\<\>\"\']*)?', re.I)
SHORTENERS = {'bit.ly', 'tinyurl.com', 't.co', 'cutt.ly', 'rb.gy'}
SUSPICIOUS_TLDS = {'xyz', 'top', 'tk', 'ml', 'buzz', 'work', 'click', 'rest', 'icu', 'club', 'monster', 'online', 'cfd'}
BRANDS = ['vodafone', 'orange', 'etisalat', 'we', 'instapay', 'fawry', 'cib', 'nbe', 'banquemisr', 'paypal', 'apple', 'amazon', 'carrefour']

def extract_urls(text):
    matches = URL_PATTERN.findall(text)
    urls = []
    for m in matches:
        m = m.strip('.,;:!?')
        if '.' in m and not re.match(r'^\d+\.\d+$', m):
            if not m.startswith(('http://', 'https://')): m = 'https://' + m
            if m not in urls: urls.append(m)
    return urls

def extract_url_features(url_str):
    if not url_str.startswith(('http://', 'https://')): url_str = 'https://' + url_str
    parsed = urlparse(url_str)
    host = (parsed.hostname or '').lower()
    path = parsed.path or ''
    is_ip = 1 if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$', host) else 0
    is_puny = 1 if 'xn--' in host else 0
    is_short = 1 if any(s in host for s in SHORTENERS) else 0
    parts = host.split('.')
    tld = parts[-1] if len(parts) > 1 else ''
    is_bad_tld = 1 if tld in SUSPICIOUS_TLDS else 0
    brands = [b for b in BRANDS if b in host or b in path.lower()]
    has_brand = 1 if len(brands) > 0 else 0
    subdomains = max(0, len(parts) - 2) if len(parts) >= 2 else 0
    
    # Shannon Entropy
    probs = [float(host.count(c)) / len(host) for c in set(host)] if host else [0]
    entropy = -sum(p * math.log2(p) for p in probs) if host else 0.0
    
    risk = 0.0
    factors = []
    if is_ip: risk += 3.5; factors.append('الرابط يستخدم عنوان IP مباشر بدلاً من نطاق رسمي')
    if is_puny: risk += 3.0; factors.append('رصد هجوم تماثل حروف (Punycode Homograph)')
    if is_short: risk += 1.8; factors.append('رابط مختصر لإخفاء الوجهة الحقيقية')
    if is_bad_tld: risk += 2.0; factors.append(f'امتداد النطاق (.{tld}) مشهور بالاحتيال')
    if has_brand and not (host.endswith(f'.{brands[0]}.com') or host == f'{brands[0]}.com'):
        risk += 2.5; factors.append(f'محاولة انتحال علامة تجارية ({brands[0]}) في نطاق غير رسمي')
    if not url_str.startswith('https://'): risk += 0.8; factors.append('الرابط يفتقر إلى تشفير HTTPS')
    
    norm_score = min(risk / 6.0, 1.0)
    return {'host': host, 'score': round(norm_score, 3), 'factors': factors, 'features': [len(url_str), len(host), is_ip, is_puny, is_short, is_bad_tld, has_brand, subdomains, entropy]}

# --- Datasets & Training ---
texts = [
    ('مبروك كسبت 50000 جنيه من فودافون كاش! دوس ع اللينك: http://voda-cash-win.xyz/claim', 1),
    ('تم ايقاف محفظتك الذكية في انستاباي. ادخل هنا وحدث بياناتك: http://instapay-security.top', 1),
    ('تنبيه البنك الاهلي: حسابك البنكي معلق. اضغط عالرابط لفك الحظر: http://192.168.1.45/nbe', 1),
    ('الحق عرض كارفور مصر! قسيمة شراء 3000 جنيه ادخل وسجل: http://carrefour-voucher.buzz', 1),
    ('شحنتك معلقة في مكتب التوزيع لسداد رسوم 25 جنيه: http://noon-track.click/pay', 1),
    ('URGENT: Your PayPal account is suspended. Verify credentials: http://paypal-verify.top', 1),
    ('Netflix subscription payment failed. Update billing: http://netflix-update.xyz', 1),
    ('فودافون كاش: تم تحويل مبلغ 1500 جنيه بنجاح. رصيدك الحالي هو 3200 جنيه.', 0),
    ('انستاباي: رمز التأكيد السري الخاص بك هو 749281. تحذير: لا تشارك هذا الرمز مع أي شخص.', 0),
    ('البنك الأهلي المصري: تم سحب مبلغ 500 جنيه من ماكينة ATM. الرصيد المتاح 18400 جنيه.', 0),
    ('جامعة القاهرة: بدء تسجيل المقررات الدراسية للفصل الثاني عبر البوابة الرسمية.', 0),
    ('Your Google verification code is 849201. Don\'t share this code with anyone.', 0),
    ('GitHub: Security alert - dependencies updated in repository cybershield.', 0)
]

X_raw = [t[0] for t in texts]
y = np.array([t[1] for t in texts])

vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), sublinear_tf=True))
])
X_vec = vectorizer.fit_transform([normalize_arabic(t) for t in X_raw])
text_clf = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
text_clf.fit(X_vec, y)

# Evaluate Model
preds = text_clf.predict(X_vec)
acc = accuracy_score(y, preds)
f1 = f1_score(y, preds)
print('===========================================================')
print(f'✅ تم تدريب نموذج النصوص بنجاح | الدقة (Accuracy): {acc*100:.1f}% | F1-Score: {f1:.4f}')
print('===========================================================')

def analyze_cyber_threat(raw_input):
    text = raw_input.strip()
    urls = extract_urls(text)
    lang = detect_language(text)
    
    # Heuristics
    norm = normalize_arabic(text)
    h_score = 0.0
    factors = []
    for pat, w, desc in SUSPICIOUS_ARABIC_PATTERNS:
        if re.search(pat, norm, re.I):
            h_score += w
            factors.append(desc)
    for pat, w, desc in SUSPICIOUS_ENGLISH_PATTERNS:
        if re.search(pat, text.lower(), re.I):
            h_score += w
            factors.append(desc)
    for pat, w, desc in LEGITIMATE_SAFETY_PATTERNS:
        if re.search(pat, norm, re.I) or re.search(pat, text.lower(), re.I):
            h_score = max(0.0, h_score - w)
            factors.append(f'مؤشر أمان: {desc}')
            
    h_norm = min(h_score / 6.0, 1.0)
    vec = vectorizer.transform([norm])
    ml_prob = float(text_clf.predict_proba(vec)[0][1])
    text_score = max(h_norm, (ml_prob * 0.6) + (h_norm * 0.4))
    
    # URL Analysis
    url_score = 0.0
    for u in urls:
        u_res = extract_url_features(u)
        url_score = max(url_score, u_res['score'])
        factors.extend(u_res['factors'])
        
    # Fusion
    if len(urls) > 0 and len(text.replace(urls[0],'').strip()) < 3:
        final_score = url_score
    elif len(urls) == 0:
        final_score = text_score
    else:
        final_score = max(url_score, (0.55 * text_score) + (0.45 * url_score))
        
    final_score = round(min(max(final_score, 0.0), 1.0), 3)
    verdict = 'Phishing' if final_score >= 0.70 else ('Suspicious' if final_score >= 0.35 else 'Safe')
    
    return {
        'verdict': verdict,
        'risk_percentage': int(final_score * 100),
        'dialect': lang['language'],
        'is_egyptian': lang['is_egyptian'],
        'urls': urls,
        'factors': list(dict.fromkeys(factors))
    }

print('🚀 محرك الفحص جاهز للعمل الفوري!')

## 3️⃣ المفحص التفاعلي الفوري داخل Colab (Interactive Scanner Widget)

In [ ]:
#@title فحص رسالة أو رابط تجريبي تفاعلياً
import ipywidgets as widgets
from IPython.display import display, HTML

preset_options = [
    ("اختر نموذج تجريبي أو اكتب بنفسك بالأسفل", ""),
    ("🇪🇬 فودافون كاش (احتيال مصري)", "مبروك كسبت 50000 جنيه من فودافون كاش! دوس ع اللينك: http://voda-cash-win.xyz/claim"),
    ("🇪🇬 انستاباي (تجميد المحفظة)", "تم ايقاف محفظتك الذكية في انستاباي. ادخل هنا وحدث بياناتك: http://instapay-security.top"),
    ("🌐 PayPal Account Alert (EN)", "URGENT: Your PayPal account is suspended. Verify credentials: http://paypal-verify.top"),
    ("✅ رسالة بنكية رسمية (آمنة)", "فودافون كاش: تم تحويل مبلغ 1500 جنيه بنجاح. رصيدك الحالي هو 3200 جنيه."),
    ("✅ كود تأكيد OTP رسمي", "انستاباي: رمز التأكيد السري الخاص بك هو 749281. تحذير: لا تشارك هذا الرمز مع أي شخص."),
    ("🔗 رابط IP مباشر مشبوه", "http://192.168.1.100/bank-login.php")
]

dropdown = widgets.Dropdown(options=preset_options, description='نماذج جاهزة:', layout=widgets.Layout(width='90%'))
text_area = widgets.Textarea(placeholder='أدخل نص الرسالة أو الرابط هنا...', layout=widgets.Layout(width='90%', height='100px'))
button = widgets.Button(description='🛡️ فحص التهديد بواسطة الذكاء الاصطناعي', button_style='danger', layout=widgets.Layout(width='90%'))
output = widgets.Output()

def on_dropdown_change(change):
    if change['new']:
        text_area.value = change['new']
dropdown.observe(on_dropdown_change, names='value')

def on_button_click(b):
    with output:
        output.clear_output()
        val = text_area.value.strip()
        if not val:
            print('الرجاء إدخال نص أولاً.')
            return
        res = analyze_cyber_threat(val)
        color = '#ff3366' if res['verdict'] == 'Phishing' else ('#f59e0b' if res['verdict'] == 'Suspicious' else '#10b981')
        title_ar = 'رصد هجوم تصيد احتيالي (Phishing)' if res['verdict'] == 'Phishing' else ('تحذير: رسالة مشبوهة' if res['verdict'] == 'Suspicious' else 'آمن: لم يتم رصد أي تهديد')
        
        factors_html = ''.join([f'<li style="margin-bottom:6px;">{f}</li>' for f in res['factors']]) if res['factors'] else '<li>رسالة نظيفة تماماً ولا تحتوي على روابط مشبوهة أو لغة إلحاح.</li>'
        
        html = f'''
        <div style="direction:rtl; font-family:sans-serif; background:#0c1527; color:#fff; padding:20px; border-radius:12px; border:2px solid {color}; margin-top:15px;">
            <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:1px solid rgba(255,255,255,0.1); padding-bottom:12px;">
                <h2 style="color:{color}; margin:0;">{title_ar}</h2>
                <div style="background:{color}; color:#fff; font-weight:bold; padding:4px 12px; border-radius:8px; font-size:18px;">
                    نسبة الخطورة: {res['risk_percentage']}%
                </div>
            </div>
            <p style="margin-top:12px; font-size:15px;"><strong>اللغة / اللهجة المكتشفة:</strong> {'مصري عامي (Egyptian Dialect)' if res['is_egyptian'] else res['dialect']}</p>
            <p><strong>الروابط المكتشفة:</strong> {', '.join(res['urls']) if res['urls'] else 'لا توجد روابط'}</p>
            <h4 style="color:#00e5ff; margin-bottom:8px;">🔍 أسباب التحذير التفسيرية (XAI):</h4>
            <ul style="color:#cbd5e1; padding-right:20px;">
                {factors_html}
            </ul>
        </div>
        '''
        display(HTML(html))

button.on_click(on_button_click)
display(dropdown, text_area, button, output)

## 4️⃣ تشغيل الموقع كاملاً برابط خارجي (Launch Full Web Dashboard)
يقوم الكود أدناه بتشغيل خادم FastAPI وموقع الويب ثنائي اللغة كاملًا، مع إتاحة رابط عام مجاني عبر `localtunnel`.

In [ ]:
#@title تشغيل موقع الويب الكامل مع رابط خارجي (Public URL)
import subprocess, time

# Install localtunnel for external access
!npm install -g localtunnel > /dev/null 2>&1

print('🌐 جاري تشغيل موقع CyberShield AI...')
print('1. سيظهر لك رابط خارجي آمن في الأسفل (your url is: https://....loca.lt)')
print('2. اضغط على الرابط لفتح لوحة التحكم ثنائية اللغة فوراً!')

# Start FastAPI server in background
server_proc = subprocess.Popen(['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)

# Run localtunnel
!npx localtunnel --port 8000